# 模型剪枝教程 (Model Pruning Tutorial)

本教程详细介绍深度学习模型剪枝技术，包括：

1. **剪枝基础**: 理解剪枝的原理和类型
2. **非结构化剪枝**: 移除单个权重
3. **结构化剪枝**: 移除整个通道/神经元
4. **重要性评估**: 不同的重要性指标
5. **迭代剪枝**: 逐步剪枝并微调

---

## 为什么需要剪枝？

研究表明，神经网络中存在大量冗余参数：

- 90% 的参数可能对最终预测贡献很小
- 剪枝可以减少模型大小 5-10 倍
- 同时保持接近原始精度

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch 版本: {torch.__version__}")

## 1. 剪枝基础

### 1.1 剪枝类型

```
非结构化剪枝              结构化剪枝
┌─────────┐              ┌───────┐
│●○●○●●○●│              │●●●●●●●│
│●○●○●●○●│   vs         │●●●●●●●│
│●○●○●●○●│              │●●●●●●●│
└─────────┘              └───────┘
  稀疏矩阵                 规则小矩阵
```

In [ ]:
# 定义测试模型
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = ConvNet()

# 统计参数
total_params = sum(p.numel() for p in model.parameters())
print(f"模型总参数量: {total_params:,}")

In [ ]:
# 可视化权重分布
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

layers = [('conv1', model.conv1.weight), ('conv2', model.conv2.weight),
          ('fc1', model.fc1.weight), ('fc2', model.fc2.weight)]

for ax, (name, weight) in zip(axes.flatten(), layers):
    w = weight.detach().flatten().numpy()
    ax.hist(w, bins=50, alpha=0.7, color='blue')
    ax.axvline(x=0, color='red', linestyle='--', label='零点')
    ax.set_title(f'{name} 权重分布 (参数量: {weight.numel():,})')
    ax.set_xlabel('权重值')
    ax.set_ylabel('频数')
    ax.legend()

plt.tight_layout()
plt.show()

print("\n观察: 大部分权重集中在零附近，这些小权重可能可以被剪枝!")

## 2. 非结构化剪枝

非结构化剪枝移除单个权重，产生稀疏矩阵。

**优点**: 高压缩率
**缺点**: 需要特殊硬件支持才能加速

In [ ]:
from pruning import (
    MagnitudePruner, PruningConfig, PruningType,
    compute_magnitude_importance, create_pruning_mask,
    compute_model_sparsity
)

# 创建新模型
model_unstructured = ConvNet()

# 配置非结构化剪枝
config = PruningConfig(
    pruning_type=PruningType.UNSTRUCTURED,
    sparsity=0.5,  # 剪枝 50% 的权重
    global_pruning=False  # 每层独立剪枝
)

pruner = MagnitudePruner(config)
pruned_model = pruner.prune(model_unstructured)

# 检查稀疏度
sparsity_stats = compute_model_sparsity(pruned_model)
print(f"整体稀疏度: {sparsity_stats['overall']:.2%}")
print(f"\n各层稀疏度:")
for name, sp in sparsity_stats['layers'].items():
    if sp > 0:
        print(f"  {name}: {sp:.2%}")

In [ ]:
# 可视化剪枝效果
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 原始权重
original_weight = model.conv1.weight.detach()[0, 0].numpy()
im1 = axes[0].imshow(original_weight, cmap='RdBu', vmin=-0.5, vmax=0.5)
axes[0].set_title('原始 Conv1 权重 (第一个滤波器)')
plt.colorbar(im1, ax=axes[0])

# 剪枝后权重
pruned_weight = pruned_model.conv1.weight.detach()[0, 0].numpy()
im2 = axes[1].imshow(pruned_weight, cmap='RdBu', vmin=-0.5, vmax=0.5)
axes[1].set_title('剪枝后 Conv1 权重 (50% 稀疏)')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# 测试剪枝模型的推理
x_test = torch.randn(16, 1, 28, 28)

with torch.no_grad():
    original_output = model(x_test)
    pruned_output = pruned_model(x_test)

# 比较输出
diff = (original_output - pruned_output).abs()
print(f"输出差异:")
print(f"  平均差异: {diff.mean():.4f}")
print(f"  最大差异: {diff.max():.4f}")

# 预测一致性
orig_pred = original_output.argmax(dim=1)
pruned_pred = pruned_output.argmax(dim=1)
consistency = (orig_pred == pruned_pred).float().mean()
print(f"\n预测一致率: {consistency * 100:.1f}%")

## 3. 结构化剪枝

结构化剪枝移除整个结构（通道、神经元），产生规则的小模型。

**优点**: 无需特殊硬件，直接获得加速
**缺点**: 压缩率相对较低

In [ ]:
from pruning import StructuredPruner

# 创建新模型
model_structured = ConvNet()

# 结构化剪枝
structured_pruner = StructuredPruner()
pruned_structured = structured_pruner.prune(model_structured, sparsity=0.3)

# 检查稀疏度
sparsity_stats = compute_model_sparsity(pruned_structured)
print(f"整体稀疏度: {sparsity_stats['overall']:.2%}")

# 获取被剪枝的通道
pruned_indices = structured_pruner.get_pruned_indices(pruned_structured)
print(f"\n被剪枝的通道数:")
for name, indices in pruned_indices.items():
    print(f"  {name}: {len(indices)} 个通道")

In [ ]:
# 可视化通道重要性
conv1_weight = model.conv1.weight.detach()
channel_importance = structured_pruner.compute_channel_importance(conv1_weight)

plt.figure(figsize=(12, 4))
plt.bar(range(len(channel_importance)), channel_importance.numpy(), alpha=0.7)
plt.axhline(y=channel_importance.quantile(0.3).item(), color='red', 
            linestyle='--', label='30% 阈值')
plt.xlabel('通道索引')
plt.ylabel('重要性 (L1 范数)')
plt.title('Conv1 各通道重要性')
plt.legend()
plt.show()

print("红线以下的通道将被剪枝")

## 4. 重要性评估方法

不同的重要性评估方法：

| 方法 | 公式 | 优点 | 缺点 |
|:-----|:-----|:-----|:-----|
| 幅度 | $\|w\|$ | 简单高效 | 可能不准确 |
| 梯度 | $\|w \cdot g\|$ | 考虑训练信息 | 需要额外计算 |
| Taylor | $\|w \cdot g\| + \frac{1}{2}h w^2$ | 理论基础强 | 计算开销大 |

In [ ]:
from pruning import (
    compute_magnitude_importance,
    compute_gradient_importance,
    compute_taylor_importance
)

# 创建示例权重和梯度
weight = torch.randn(64, 32)
gradient = torch.randn(64, 32)

# 计算不同的重要性
mag_importance = compute_magnitude_importance(weight, dim=None)
grad_importance = compute_gradient_importance(weight, gradient, dim=None)
taylor_importance = compute_taylor_importance(weight, gradient, dim=None)

# 可视化比较
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, imp) in zip(axes, [
    ('幅度重要性', mag_importance),
    ('梯度重要性', grad_importance),
    ('Taylor 重要性', taylor_importance)
]):
    im = ax.imshow(imp.numpy(), cmap='hot', aspect='auto')
    ax.set_title(name)
    ax.set_xlabel('输入维度')
    ax.set_ylabel('输出维度')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# 比较不同方法选择的剪枝位置
sparsity = 0.5

mask_mag = create_pruning_mask(mag_importance, sparsity, structured=False)
mask_grad = create_pruning_mask(grad_importance, sparsity, structured=False)
mask_taylor = create_pruning_mask(taylor_importance, sparsity, structured=False)

# 计算重叠率
overlap_mag_grad = ((mask_mag == 0) & (mask_grad == 0)).float().mean()
overlap_mag_taylor = ((mask_mag == 0) & (mask_taylor == 0)).float().mean()
overlap_grad_taylor = ((mask_grad == 0) & (mask_taylor == 0)).float().mean()

print("不同方法剪枝位置的重叠率:")
print(f"  幅度 vs 梯度: {overlap_mag_grad:.2%}")
print(f"  幅度 vs Taylor: {overlap_mag_taylor:.2%}")
print(f"  梯度 vs Taylor: {overlap_grad_taylor:.2%}")

## 5. 全局剪枝 vs 局部剪枝

**局部剪枝**: 每层独立剪枝相同比例
**全局剪枝**: 所有层一起排序，剪枝最不重要的

In [ ]:
# 局部剪枝
model_local = ConvNet()
config_local = PruningConfig(sparsity=0.5, global_pruning=False)
pruner_local = MagnitudePruner(config_local)
pruned_local = pruner_local.prune(model_local)

# 全局剪枝
model_global = ConvNet()
config_global = PruningConfig(sparsity=0.5, global_pruning=True)
pruner_global = MagnitudePruner(config_global)
pruned_global = pruner_global.prune(model_global)

# 比较各层稀疏度
local_stats = compute_model_sparsity(pruned_local)
global_stats = compute_model_sparsity(pruned_global)

print("局部剪枝 vs 全局剪枝 各层稀疏度:")
print(f"{'层名':<15} {'局部剪枝':<12} {'全局剪枝':<12}")
print("-" * 40)
for name in local_stats['layers']:
    local_sp = local_stats['layers'].get(name, 0)
    global_sp = global_stats['layers'].get(name, 0)
    if local_sp > 0 or global_sp > 0:
        print(f"{name:<15} {local_sp:<12.2%} {global_sp:<12.2%}")

## 6. 剪枝后微调

剪枝后通常需要微调来恢复精度。

In [ ]:
from pruning import prune_model

# 创建模型并模拟训练
model_finetune = ConvNet()

# 模拟训练数据
train_data = [(torch.randn(32, 1, 28, 28), torch.randint(0, 10, (32,))) 
              for _ in range(50)]

# 剪枝
pruned_model = prune_model(model_finetune, sparsity=0.5, pruning_type="unstructured")

# 微调
optimizer = torch.optim.Adam(pruned_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

print("微调剪枝模型...")
pruned_model.train()
for epoch in range(3):
    total_loss = 0
    for x_batch, y_batch in train_data:
        optimizer.zero_grad()
        output = pruned_model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"  Epoch {epoch+1}: Loss = {total_loss/len(train_data):.4f}")

print("\n微调完成!")

## 7. 不同稀疏度的影响

探索不同稀疏度对模型的影响。

In [ ]:
# 测试不同稀疏度
sparsities = [0.1, 0.3, 0.5, 0.7, 0.9]
results = []

x_test = torch.randn(32, 1, 28, 28)
original_model = ConvNet()
original_model.eval()

with torch.no_grad():
    original_output = original_model(x_test)

for sp in sparsities:
    # 剪枝
    model_test = ConvNet()
    model_test.load_state_dict(original_model.state_dict())
    pruned = prune_model(model_test, sparsity=sp)
    
    # 测试
    pruned.eval()
    with torch.no_grad():
        pruned_output = pruned(x_test)
    
    # 计算指标
    diff = (original_output - pruned_output).abs().mean().item()
    pred_match = (original_output.argmax(1) == pruned_output.argmax(1)).float().mean().item()
    
    results.append({
        'sparsity': sp,
        'output_diff': diff,
        'pred_match': pred_match
    })

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot([r['sparsity'] for r in results], 
             [r['output_diff'] for r in results], 'bo-')
axes[0].set_xlabel('稀疏度')
axes[0].set_ylabel('输出差异')
axes[0].set_title('稀疏度 vs 输出差异')

axes[1].plot([r['sparsity'] for r in results], 
             [r['pred_match'] * 100 for r in results], 'ro-')
axes[1].set_xlabel('稀疏度')
axes[1].set_ylabel('预测一致率 (%)')
axes[1].set_title('稀疏度 vs 预测一致率')
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

## 总结

本教程介绍了模型剪枝的核心概念：

1. **非结构化剪枝**: 高压缩率，需要特殊硬件
2. **结构化剪枝**: 直接加速，压缩率较低
3. **重要性评估**: 幅度、梯度、Taylor
4. **全局 vs 局部**: 不同的剪枝策略
5. **微调**: 恢复剪枝后的精度

### 选择建议

- **需要实际加速**: 结构化剪枝
- **追求高压缩率**: 非结构化剪枝 + 稀疏计算
- **精度敏感**: 迭代剪枝 + 微调